# Task 1: EDA — Traffic Speed Prediction with Event Text

Explore the BjTT dataset: speed time series, event text, road network, adjacency.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from collections import Counter
from pathlib import Path

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency, load_roads, build_windows, compute_norm_stats,
    DATASET_DIR,
)

## 1. Train Speed Data

In [ ]:
s1, s2 = load_train_speeds()
print(f"Block 1: {s1.shape}, dtype={s1.dtype}, range=[{s1.min():.1f}, {s1.max():.1f}]")
print(f"Block 2: {s2.shape}, dtype={s2.dtype}, range=[{s2.min():.1f}, {s2.max():.1f}]")

In [ ]:
all_speeds = np.concatenate([s1, s2], axis=0)
print(f"Total timesteps: {all_speeds.shape[0]}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(all_speeds.flatten(), bins=80, edgecolor='black', alpha=0.7)
axes[0].set_title("Speed distribution (all roads, all time)")
axes[0].set_xlabel("km/h")

road_means = all_speeds.mean(axis=0)
axes[1].hist(road_means, bins=40, edgecolor='black', alpha=0.7)
axes[1].set_title("Per-road mean speed")
axes[1].set_xlabel("km/h")

road_stds = all_speeds.std(axis=0)
axes[2].hist(road_stds, bins=40, edgecolor='black', alpha=0.7)
axes[2].set_title("Per-road std speed")
axes[2].set_xlabel("km/h")

plt.tight_layout()
plt.show()

In [ ]:
zero_frac = (all_speeds == 0).mean(axis=0)
print(f"Roads with >50% zeros: {(zero_frac > 0.5).sum()} / 1260")
print(f"Roads with any zeros: {(zero_frac > 0).sum()} / 1260")

In [ ]:
roads_subset = [0, 1, 2, 10, 50, 100, 500, 1000]
fig, axes = plt.subplots(len(roads_subset), 1, figsize=(16, 12), sharex=True)
for i, r in enumerate(roads_subset):
    axes[i].plot(all_speeds[:, r], linewidth=0.5)
    axes[i].set_ylabel(f"Road {r}")
axes[-1].set_xlabel("Timestep")
fig.suptitle("Speed time series — sample roads")
plt.tight_layout()
plt.show()

## 2. Event Text

In [ ]:
t1, t2 = load_train_texts()
print(f"Block 1 texts: {len(t1)}, Block 2 texts: {len(t2)}")
print(f"Sample text: {t1[0][:300]}...")

In [ ]:
all_text_lens = [len(t.split()) for t in t1 + t2]
print(f"Text length (words): min={min(all_text_lens)}, max={max(all_text_lens)}, mean={np.mean(all_text_lens):.1f}")

unique_texts = set(t1 + t2)
print(f"Unique texts: {len(unique_texts)} / {len(t1) + len(t2)} total")

plt.hist(all_text_lens, bins=50, edgecolor='black', alpha=0.7)
plt.title("Event text length distribution (words)")
plt.xlabel("Word count")
plt.show()

In [ ]:
all_words = " ".join(t1 + t2).lower().replace(".", "").split()
word_freq = Counter(all_words)
print("Most common words:")
for w, c in word_freq.most_common(30):
    print(f"  {w}: {c}")

## 3. Road Network

In [ ]:
adj = load_adjacency()
print(f"Adjacency: {adj.shape}, dtype={adj.dtype}")
edges = np.count_nonzero(adj)
print(f"Edges: {edges} — {edges / (1260*1260) * 100:.3f}% dense")

degrees = adj.sum(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(degrees, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title("Node degree distribution")
axes[0].set_xlabel("Degree")

axes[1].bar(range(len(degrees)), sorted(degrees, reverse=True), width=1, alpha=0.7)
axes[1].set_title("Degree rank plot")
axes[1].set_xlabel("Rank")
axes[1].set_ylabel("Degree")
plt.tight_layout()
plt.show()

print(f"Isolated nodes (degree=0): {(degrees == 0).sum()}")
print(f"Max degree: {degrees.max()}")

In [ ]:
roads = load_roads()
print(f"Roads: {len(roads)} entries")
print(f"road[0]: {roads[0]}")
print(f"road[0] has {len(roads[0])} sub-segments")

roadclasses = Counter()
sub_seg_counts = []
for r in roads:
    sub_seg_counts.append(len(r))
    for seg in r:
        roadclasses[seg.get('roadclass', -1)] += 1

print(f"\nRoad classes: {roadclasses.most_common()}")
print(f"Sub-segments per road: min={min(sub_seg_counts)}, max={max(sub_seg_counts)}, mean={np.mean(sub_seg_counts):.1f}")

## 4. Train Windows (Supervised Samples)

In [ ]:
X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)
print(f"Block 1 windows: {X1.shape}, targets: {Y1.shape}")
print(f"Block 2 windows: {X2.shape}, targets: {Y2.shape}")
print(f"Total train windows: {len(X1) + len(X2)}")

In [ ]:
mean, std = compute_norm_stats(np.concatenate([s1, s2], axis=0))
print(f"Norm mean range: [{mean.min():.2f}, {mean.max():.2f}]")
print(f"Norm std range: [{std.min():.2f}, {std.max():.2f}]")

## 5. Test Data

In [ ]:
test_hist, test_texts = load_test_data()
print(f"Test hist: {test_hist.shape}, dtype={test_hist.dtype}")
print(f"Test range: [{test_hist.min():.1f}, {test_hist.max():.1f}]")
print(f"Test texts: {len(test_texts)}")
print(f"Sample text: {test_texts[0][:300]}...")

In [ ]:
sub = pd.read_csv(DATASET_DIR / "sample_submission.csv")
print(f"Submission rows: {len(sub)}")
print(f"Expected: 540 samples x 3 horizons x 1260 roads = {540 * 3 * 1260:,}")
print(f"Sample ids:\n{sub.head(10)}")